# varsplit — splitting protein variant datasets for machine learning

When training a model on protein fitness data, how you split your data into training and test sets matters enormously. A random split looks clean on paper, but it can give a falsely optimistic picture of how well your model will actually generalize — because the test set ends up looking almost identical to the training set.

**varsplit** provides splitting strategies that reflect real biological questions:
- Can the model generalize to *new positions* it has never seen mutated?
- Can it predict *higher-order mutants* from single-mutant training data?
- Can it extrapolate toward *high-fitness variants* it was not trained on?

This notebook walks through each strategy using a real deep mutational scanning datasetin the ProteinGym: DMS ID **CBPA2_HUMAN_Tsuboyama_2023_1O6X**  from [Tsuboyama et al. 2023](https://www.nature.com/articles/s41586-023-06328-6). The dataset contains 1,357 single mutants and 711 double mutants, with experimentally measured stability scores.

---

## Setup

Install varsplit directly from GitHub if you haven't already:

```bash
pip install git+https://github.com/TCoulth/varsplit.git
```

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from varsplit import train_test_split, KFold
from varsplit.diagnose import Diagnose

In [ ]:
%matplotlib inline

## Load the data

The dataset is included alongside this notebook. It contains both single and double mutants in a single file.

The mutation strings use standard notation: `A24V` means alanine at position 24 mutated to valine. Double mutants are colon-separated: `E60A:Y66C`.

In [ ]:
df = pd.read_csv("CBPA2_HUMAN_Tsuboyama_2023_1O6X.csv")

print(f"Total variants : {len(df)}")
print(f"  Singles      : {(df['mutation_depth'] == 1).sum()}")
print(f"  Doubles      : {(df['mutation_depth'] == 2).sum()}")
print(f"Positions      : 1-72")
print(f"DMS score range: {df['DMS_score'].min():.3f} to {df['DMS_score'].max():.3f}")
df.head()

---
## Strategy 1: Random split

The standard baseline. Variants are randomly assigned to train or test with no regard for their biological properties. This is the default in most ML workflows.

It is useful as a **lower bound on difficulty** — if your model can't beat random-split performance, something is fundamentally wrong. But strong random-split performance doesn't tell you much about real-world generalization.

The plots below shows the difference in properties between the train and test sets. In the case of random splitting, the properties mirror each other. For example, the mutation order distribution is identical in train and test — by design, the two sets are statistically indistinguishable.

In [ ]:
train_idx, test_idx = train_test_split(
    df,
    strategy="random",
    mutation_col="mutant",
    test_size=0.2,
    random_state=42,
)

train_df = df.iloc[train_idx]
test_df  = df.iloc[test_idx]

In [ ]:
diag = Diagnose(train_df, test_df, mutation_col="mutant", label_A="train", label_B="test", fitness_col='DMS_score')
fig  = diag.plot_order_distribution()
plt.show()

fig  = diag.plot_positional_overlap()
plt.show()

fig  = diag.plot_mutational_novelty()
plt.show()

fig = diag.plot_fitness_distributions()
plt.show()

---
## Strategy 2: Order split

Splits by the number of mutations per variant. By default, single mutants go to train and higher-order mutants go to test.

This directly asks: **can a model trained on single-mutant data predict the fitness of double mutants?** This is useful if you're workflow starts at single mutations or small numbers and builds up during a campaign.

Note that `test_size` is not used here — the split boundary is determined by mutation order, so the train/test proportions reflect the composition of your dataset.

In [ ]:
train_idx, test_idx = train_test_split(
    df,
    strategy="order",
    mutation_col="mutant",
)

train_df = df.iloc[train_idx]
test_df  = df.iloc[test_idx]

In [ ]:
diag = Diagnose(train_df, test_df, mutation_col="mutant", label_A="train", label_B="test")
fig  = diag.plot_order_distribution()
plt.show()



---
## Strategy 3: Positional split

Holds out a subset of sequence positions entirely. All variants that touch a held-out position go to test — including any double mutants where one of the two mutations is at a held-out position.

This asks: **can the model generalize to positions it has never seen mutated?** This is relevant when you want to apply a model trained on one region of a protein to predict variants in a different region, or when experimental coverage is uneven.

`test_size` here controls the fraction of *positions* held out, not variants. Because many variants may touch the same position, the actual variant fraction in the test set can differ from `test_size`.

In [ ]:
train_idx, test_idx = train_test_split(
    df,
    strategy="positional",
    mutation_col="mutant",
    test_size=0.2,
    random_state=42,
)

train_df = df.iloc[train_idx]
test_df  = df.iloc[test_idx]

In [ ]:
diag = Diagnose(train_df, test_df, mutation_col="mutant", label_A="train", label_B="test")
fig  = diag.plot_positional_overlap()
plt.show()

---
## Strategy 4: Mutational split

Holds out specific amino acid substitutions — defined as `(position, mutant_aa)` pairs. For example, if `(24, V)` is held out, then `A24V` and any double mutant containing `A24V` goes to test.

Unlike positional splitting, the *position itself* may still appear in train — just with different amino acid substitutions. This asks the question: **can the model predict the effect of a specific amino acid change at a position it has seen, but not that exact substitution?**

> **Note:** This dataset contains only single and double mutants. On a singles-only dataset, mutational splitting behaves similarly to random splitting, since there is no combinatorial structure to exploit. It becomes more informative on datasets with higher mutation orders.

In [ ]:
train_idx, test_idx = train_test_split(
    df,
    strategy="mutational",
    mutation_col="mutant",
    test_size=0.2,
    random_state=42,
)

train_df = df.iloc[train_idx]
test_df  = df.iloc[test_idx]

In [ ]:
diag = Diagnose(train_df, test_df, mutation_col="mutant", label_A="train", label_B="test")
fig  = diag.plot_mutational_novelty()
plt.show()

---
## Strategy 5: Fitness split

Splits by fitness score. By default, the top `test_size` fraction by score goes to test — so the model is trained on lower-fitness variants and evaluated on high-fitness ones.

This directly asks: **can the model extrapolate toward high-fitness variants it was not trained on?**

You can also split on the bottom tail (`upper_tail=False`) or use an absolute threshold (`threshold=0.5`) instead of a quantile.

In [ ]:
train_idx, test_idx = train_test_split(
    df,
    strategy="fitness",
    mutation_col="mutant",
    fitness_col="DMS_score",
    test_size=0.2,
)

train_df = df.iloc[train_idx]
test_df  = df.iloc[test_idx]

In [ ]:
diag = Diagnose(
    train_df, test_df,
    mutation_col="mutant",
    fitness_col="DMS_score",
    label_A="train",
    label_B="test",
)
fig = diag.plot_fitness_distributions()
plt.show()

---
## Cross-validation

All strategies also support k-fold cross-validation via `KFold`. The API mirrors scikit-learn:

In [ ]:
kf = KFold(n_splits=5, strategy="positional", mutation_col="mutant", random_state=42)

for fold, (train_idx, test_idx) in enumerate(kf.split(df)):
    train_df = df.iloc[train_idx]
    test_df  = df.iloc[test_idx]
    print(f"Fold {fold+1}: train={len(train_df)}, test={len(test_df)}")

---
## What next?

The gap between random-split and strategy-split performance on your model is a direct measure of how well it generalizes along that biological dimension. A model that performs well under random splitting but poorly under positional splitting has learned position-specific patterns — it hasn't learned anything transferable about how mutations affect protein stability in general.

varsplit also includes a `Diagnose` module for characterizing the generalization challenge between any two fixed variant sets — useful when you have a pre-defined train/test split and want to understand what biological question it is actually asking.

See the [varsplit GitHub](https://github.com/TCoulth/varsplit) for full documentation.